# Cinema Backend — kiểm tra runner GĐ2 trên Kaggle

Notebook clone branch `feature/phase2-runner-observability` từ repository hiện tại, dựng PostgreSQL và API, seed fixture, sau đó chạy nhanh S1 contention ở 2 VU × 3 lượt. Mục tiêu là xác nhận runner ghi đủ tài nguyên tiến trình, `pg_stat_activity`, deadlock trước/sau và snapshot Prometheus. API, PostgreSQL và k6 cùng chạy trong một phiên Kaggle CPU.

In [ ]:
import os, pathlib, subprocess, json, time, urllib.request

REPO_URL = 'https://github.com/taclna/BTL-1-21-phase-2.git'
REF = 'feature/phase2-runner-observability'
repo = pathlib.Path('/kaggle/working/BTL-1-21-phase-2')

if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', REF, '--single-branch',
                    REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', REF], cwd=repo, check=True)
    subprocess.run(['git', 'checkout', '-B', REF, 'origin/' + REF], cwd=repo, check=True)

os.chdir(repo)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=all',
                                 '--', 'src/main', 'pom.xml', 'load-test'], text=True).strip()
assert branch == REF, (branch, REF)
assert not dirty, 'Source/load-test phải sạch trước khi đo:\n' + dirty
print('Repository:', REPO_URL)
print('Branch:', branch)
print('Commit:', head)


In [ ]:
%%bash
set -euo pipefail
sudo apt-get update -qq
sudo apt-get install -y openjdk-21-jdk-headless postgresql postgresql-contrib nodejs curl ca-certificates gnupg
curl -fsSL https://dl.k6.io/key.gpg | sudo gpg --batch --yes --dearmor -o /usr/share/keyrings/k6-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/k6-archive-keyring.gpg] https://dl.k6.io/deb stable main' | sudo tee /etc/apt/sources.list.d/k6.list
sudo apt-get update -qq
sudo apt-get install -y k6


In [ ]:
import glob
jdk = sorted(glob.glob('/usr/lib/jvm/java-21-openjdk-*'))
assert jdk, 'Chưa cài được JDK 21'
os.environ['JAVA_HOME'] = jdk[0]
os.environ['PATH'] = jdk[0] + '/bin:' + os.environ['PATH']
subprocess.run(['java', '-version'], check=True)
subprocess.run(['k6', 'version'], check=True)
node_major = int(subprocess.check_output(['node', '-p', 'process.versions.node.split(".")[0]'], text=True))
assert node_major >= 18, 'Seed script cần Node.js >= 18 (fetch API)'


In [ ]:
clusters = subprocess.check_output(['pg_lsclusters', '--no-header'], text=True).splitlines()
assert clusters, 'Gói PostgreSQL chưa tạo cluster; kiểm tra quá trình cài đặt'
cluster = clusters[-1].split()
pg_version, pg_name, pg_port = cluster[:3]
subprocess.run(['sudo', 'pg_ctlcluster', '--skip-systemctl-redirect',
                pg_version, pg_name, 'start'], check=False)

def pg(sql):
    return subprocess.check_output(['sudo', '-u', 'postgres', 'psql', '-p', pg_port,
                                    '-v', 'ON_ERROR_STOP=1', '-tAc', sql], text=True).strip()

pg("""DO $$ BEGIN
    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname='cinema_load') THEN
        CREATE ROLE cinema_load LOGIN PASSWORD 'cinema_load_pw';
    END IF;
END $$;""")
if pg("SELECT 1 FROM pg_database WHERE datname='cinema_load'") != '1':
    subprocess.run(['sudo', '-u', 'postgres', 'createdb', '-p', pg_port,
                    '-O', 'cinema_load', 'cinema_load'], check=True)
data_dir = pg('SHOW data_directory')
database_pid = int(subprocess.check_output(['sudo', 'head', '-n', '1', data_dir + '/postmaster.pid'], text=True))
print('PostgreSQL:', pg('SELECT version()'))
print('Database PID:', database_pid, 'port:', pg_port)


In [ ]:
os.environ.update({
    'SPRING_PROFILES_ACTIVE': 'local',
    'DB_URL': f'jdbc:postgresql://127.0.0.1:{pg_port}/cinema_load',
    'DB_USERNAME': 'cinema_load', 'DB_PASSWORD': 'cinema_load_pw',
    'PGHOST': '127.0.0.1', 'PGPORT': str(pg_port),
    'PGDATABASE': 'cinema_load', 'PGUSER': 'cinema_load',
    'PGPASSWORD': 'cinema_load_pw',
    'JWT_SECRET': 'kaggle-local-only-cinema-benchmark-secret-2026-please-change',
    'DEMO_ADMIN_USERNAME': 'admin', 'DEMO_ADMIN_PASSWORD': 'demo-pass-1234',
    'JWT_EXPIRATION_SECONDS': '14400',
    'SERVER_ADDRESS': '127.0.0.1', 'SERVER_PORT': '8080',
    'BASE_URL': 'http://127.0.0.1:8080',
    'PROMETHEUS_URL': 'http://127.0.0.1:8080/actuator/prometheus',
    'ADMIN_USERNAME': 'admin', 'ADMIN_PASSWORD': 'demo-pass-1234',
})
subprocess.run(['bash', './mvnw', '-B', '-DskipTests', 'package'], check=True)
jars = list((repo / 'target').glob('*.jar'))
assert len(jars) == 1, jars
api_log = open('/kaggle/working/cinema-api.log', 'w')
api = subprocess.Popen(['java', '-Xms256m', '-Xmx768m', '-jar', str(jars[0])],
                       stdout=api_log, stderr=subprocess.STDOUT, env=os.environ.copy())
os.environ['API_PID'] = str(api.pid)
os.environ['DATABASE_PID'] = str(database_pid)
ready = False
for _ in range(120):
    if api.poll() is not None:
        break
    try:
        with urllib.request.urlopen(os.environ['BASE_URL'] + '/actuator/health/readiness', timeout=2) as r:
            ready = r.status == 200
        if ready:
            break
    except Exception:
        pass
    time.sleep(1)
if not ready:
    print(pathlib.Path('/kaggle/working/cinema-api.log').read_text()[-12000:])
    raise RuntimeError('API chưa sẵn sàng; xử lý lỗi trước khi đo')
print('API ready, PID:', api.pid)


In [ ]:
import urllib.error
for method, route in [('GET', '/api/v1/users/me'), ('POST', '/api/v1/bookings')]:
    request = urllib.request.Request(os.environ['BASE_URL'] + route,
        data=b'{}' if method == 'POST' else None, method=method,
        headers={'Content-Type': 'application/json'})
    try:
        urllib.request.urlopen(request)
        raise AssertionError(f'{method} {route} cho truy cập khi thiếu token')
    except urllib.error.HTTPError as e:
        assert e.code == 401, (method, route, e.code)
        print(method, route, '→ 401 OK')

os.environ.update({'MOVIE_COUNT': '1000', 'DELETED_COUNT': '50',
                   'SEED_SCOPE': 'all', 'VUS': '50',
                   'FIXTURES': str(repo / 'load-test/fixtures.local.json')})
subprocess.run(['node', 'load-test/seed.mjs'], check=True)
subprocess.run(['k6', 'run', '--include-system-env-vars', 'load-test/smoke.js'], check=True)

postgres_check = subprocess.check_output(
    ['psql', '-X', '-w', '-Atc', 'select current_database(), current_user'],
    env=os.environ.copy(), text=True).strip()
with urllib.request.urlopen(os.environ['PROMETHEUS_URL'], timeout=3) as response:
    prometheus_status = response.status
    prometheus_preview = response.read(500).decode('utf-8', errors='replace')
assert postgres_check == 'cinema_load|cinema_load', postgres_check
assert prometheus_status == 200, prometheus_status
print('PostgreSQL monitor:', postgres_check)
print('Prometheus monitor: HTTP', prometheus_status)
print(prometheus_preview[:200])


In [ ]:
# Lượt xác nhận nhanh runner GĐ2: 2 VU, 3 ghế contention độc lập.
subprocess.run([
    'python', 'load-test/run-baseline.py',
    '--scenario', 'contention',
    '--levels', '2',
    '--repeats', '3',
    '--api-pid', str(api.pid),
    '--database-pid', str(database_pid),
], cwd=repo, env=os.environ.copy(), check=True)


In [ ]:
import csv

results_root = repo / 'load-test/results'
run_dirs = [path for path in results_root.iterdir() if path.is_dir()]
assert run_dirs, 'Runner chưa tạo thư mục kết quả'
run_dir = max(run_dirs, key=lambda path: path.stat().st_mtime)
print('Result directory:', run_dir)
for path in sorted(run_dir.iterdir()):
    print(f'{path.name}: {path.stat().st_size} bytes')

deadlock_files = sorted(run_dir.glob('*-deadlocks.json'))
postgres_files = sorted(run_dir.glob('*-postgres.csv'))
prometheus_files = sorted(run_dir.glob('*-prometheus.txt'))
assert len(deadlock_files) == len(postgres_files) == len(prometheus_files) == 3

for path in deadlock_files:
    evidence = json.loads(path.read_text(encoding='utf-8'))
    assert evidence['beforeError'] is None and evidence['afterError'] is None, evidence
    assert evidence['delta'] == 0, evidence
    print(path.name, 'deadlock delta =', evidence['delta'])

for path in postgres_files:
    with path.open(encoding='utf-8') as stream:
        samples = list(csv.DictReader(stream))
    failed = [row for row in samples if row['available'].lower() != 'true']
    assert samples and not failed, (path.name, failed[:3])
    print(path.name, 'samples =', len(samples), 'failed =', len(failed))

for path in prometheus_files:
    content = path.read_text(encoding='utf-8')
    sample_count = content.count('# runner_sample timestampUtc=')
    error_count = content.count('# runner_sample_error')
    assert sample_count >= 1 and error_count == 0, (path.name, sample_count, error_count)
    print(path.name, 'samples =', sample_count, 'errors =', error_count)


In [ ]:
import shutil

# Chỉ đóng gói thư mục kết quả mới nhất; không chứa fixture JWT hay mật khẩu.
archive = shutil.make_archive(
    '/kaggle/working/cinema-phase2-runner-test',
    'zip',
    root_dir=str(run_dir.parent),
    base_dir=run_dir.name,
)
print('Tải tại Output:', archive)

# Dừng API và đóng log sau khi đã thu đủ evidence.
if api.poll() is None:
    api.terminate()
    try:
        api.wait(timeout=15)
    except subprocess.TimeoutExpired:
        api.kill()
        api.wait()
api_log.close()
print('API stopped with code:', api.returncode)
